In [6]:
import sys
sys.path.insert(0, '../')
import pandas as pd
import numpy as np
import time
from datetime import datetime
import matplotlib as mpl
import matplotlib.pyplot as plt
from datetime import datetime, timedelta
import custom_funcs as cf
import price_calc as pc
import plot_funcs as pf
#import pvlib
import annual_val_funcs as avf
import os
import glob
import flats
from flats import Building
from flats import Flat
from flats import PV_system
import funcs

# Check that old files are not present
if 'df_results' in locals():
    del df_results

In [7]:
# Read files from a specific directory

#directory = './Files/2025-03-21_PVSyst_initial/Monofacial/'
directory = './Files/2025-07-01_PVsyst_data/2026-01-20_unshaded_Magda_2/'
#directory = './Files/2025-07-01_PVsyst_data/2025-03-21_PVSyst_initial/Full_set/'
forbidden_files = [] # Files that should not be processed
monofacial_identifiers = ['MPV_', '_S'] # Identifiers for monofacial files
bifacial_identifiers = ['_E', '_W'] # Identifiers for bifacial files
#bifacial_identifiers = ['_F', '_R'] # Identifiers for bifacial files

datenote = '2026-01-20_'
identifier = 'Unshaded'


In [8]:
def process_files_monofacial(directory, file, return_column='E_Grid', lin_names=11, lin_data=14, decimal='.', encoding='utf-8'):
    """
    Process a CSV file for monofacial PV systems.
    Args:
        file (str): The name of the CSV file to process.
        return_column (str): The column to return from the DataFrame.
        lin_names (int): The line number where names are located in the CSV.
        lin_data (int): The line number where data starts in the CSV.
        decimal (str): The decimal separator used in the CSV.
        encoding (str): The encoding of the CSV file.
    Returns:
        pd.Series: A pandas Series containing the specified column from the DataFrame.
    """
    if file.endswith('.CSV'):
        # Read the CSV file into a DataFrame

        names_pv = pd.read_csv(os.path.join(directory, file), skiprows=lin_names-1, nrows=1, header=None, encoding=encoding, decimal=decimal)
        names_ser = [names_pv.values[0][i] for i in range(len(names_pv.columns))]
        ser = pd.read_csv(os.path.join(directory, file), skiprows=lin_data-1, header=None, names=names_ser, index_col=0, encoding=encoding, decimal=decimal)
        #print(ser.loc[:,return_column])
        return ser[return_column]
    
def process_files_bifacial(directory, file, return_column='E_Grid', lin_names=11, lin_data=14, decimal='.', encoding='utf-8'):
    """
    Process a CSV file for bifacial PV systems.
    Args:
        file (str): The name of the CSV file to process.
        return_column (str): The column to return from the DataFrame.
        lin_names (int): The line number where names are located in the CSV.
        lin_data (int): The line number where data starts in the CSV.
        decimal (str): The decimal separator used in the CSV.
        encoding (str): The encoding of the CSV file.
    Returns:
        pd.Series: A pandas Series containing the specified column from the DataFrame.
    """
    if file.endswith('.CSV'):
        # Read the CSV file into a DataFrame
        names_pv = pd.read_csv(os.path.join(directory, file), skiprows=lin_names-1, nrows=1, header=None, encoding=encoding, decimal=decimal)
        names_ser = [names_pv.values[0][i] for i in range(len(names_pv.columns))]
        ser = pd.read_csv(os.path.join(directory, file), skiprows=lin_data-1, header=None, names=names_ser, index_col=0, encoding=encoding, decimal=decimal)

        if file.endswith('_E.CSV'):
            file2 = file.replace('_E.CSV', '_W.CSV')
        elif file.endswith('_W.CSV'):
            file2 = file.replace('_W.CSV', '_E.CSV')
        elif file.endswith('_F.CSV'):
            file2 = file.replace('_F.CSV', '_R.CSV')
        elif file.endswith('_R.CSV'):
            file2 = file.replace('_R.CSV', '_F.CSV')

        names_pv2 = pd.read_csv(os.path.join(directory, file2), skiprows=lin_names-1, nrows=1, header=None, encoding=encoding, decimal=decimal)
        names_ser2 = [names_pv2.values[0][i] for i in range(len(names_pv2.columns))]
        
        ser2 = pd.read_csv(os.path.join(directory, file2), skiprows=lin_data-1, header=None, names=names_ser2, index_col=0, encoding=encoding, decimal=decimal)
        
        if ser[return_column].sum(axis=0) > ser2[return_column].sum(axis=0):
            ret_col = ser[return_column] + 0.9*ser2[return_column]
        else:
            ret_col = ser2[return_column] + 0.9*ser[return_column]
            #print('Bifacial PV system')
        ret_col.name = return_column
    
        return ret_col, file2
    
    


In [ ]:
files = os.listdir(directory)
save_df = True  # Save the DataFrame to an Excel file
#if directory.__contains__('monofacial'):
# Initialize empty dataframe
if 'df_results' not in locals():
    df_results = pd.DataFrame(index=pd.date_range(start='2023-01-01 00:00:00', periods=8760, freq='h'))

for file in files:

    if (file.endswith('.CSV') & (file not in forbidden_files)):

        if any(sub in file for sub in monofacial_identifiers):
            # Process the file using the monofacial method        
            prof = process_files_monofacial(directory, file)
            col_name = file.replace('.CSV', '_monofacial')

        elif any(sub in file for sub in bifacial_identifiers):
            # Process the file using the bifacial method
            prof, file2 = process_files_bifacial(directory, file)
            col_name = file.replace('.CSV', '_bifacial')
            forbidden_files.append(file2)

        else:
            print(f"File {file} does not match any processing criteria.")
            continue
            
        prof.index = df_results.index
        df_results[col_name] = prof
       
print(df_results)
fig, ax = plt.subplots(figsize=(10, 6))
ax.plot(df_results.loc['2023-06-10':'2023-06-11',:], label=df_results.columns)
ax.set_ylabel('Power (kW)')
ax.set_xlabel('Time')
ax.grid()
ax.legend(loc='upper left', bbox_to_anchor=(1, 1))
plt.tight_layout()

print(df_results.sum(axis=0))

if save_df:
    df_results.to_excel(datenote + identifier + '.xlsx')
    print('DataFrame saved to excel file.')
    fig.savefig(datenote + identifier + '.jpeg', dpi=300, bbox_inches='tight')
       